In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import os
from math import ceil
from scipy.stats import gaussian_kde
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import random


try:
    import umap
    HAS_UMAP = True
except ImportError:
    HAS_UMAP = False

In [2]:
def sample_duplicates_test(df):
    grouped = df.groupby("sample_id")
    for name, group in grouped:
        if (len(group) > 1):
            print(f"Sample ID: {name}")
            print(group)
            break  # Just show the first group for brevity

In [3]:
geneticsData = pd.read_csv('../data/GNPC_Harmonized_Dataset_V1/GeneticsV1_anonymized.csv')
geneticsData.head()

,contributor_code,sample_id,gene,variant
0,R,6cafef94-c7c7-4b94-a2db-001b1e9bc258,APOE,34
1,C,69e4c6ae-4523-4146-aebb-001d13bb5146,APOE,32
2,F,aa2f62aa-c0d3-40f8-9d51-0077c2ff098e,APOE,33
3,Q,ab15806d-4c39-482d-a91c-301972a0c4b5,APOE,44
4,H,968c7201-f58f-41a9-a2b4-00db0c972e19,APOE,33


In [4]:
geneticsData.gene.unique()

array(['APOE', 'C9', 'GRN', 'MAPT', 'C9ORF72', 'ANG', 'FUS', 'SETX',
       'VAPB', 'SOD1', 'VCP', 'TAU', 'TDP43', 'PROGRAN'], dtype=object)

In [5]:
geneticsData.variant.unique()

array([34, 32, 33, 44, 23, -1, 24,  0, 22,  1])

In [6]:
sample_duplicates_test(geneticsData)

Sample ID: 001886bf-3b1f-49cd-a7fc-4a1ec3f765c1
     contributor_code                             sample_id  gene  variant
8328                N  001886bf-3b1f-49cd-a7fc-4a1ec3f765c1  MAPT        0
8329                N  001886bf-3b1f-49cd-a7fc-4a1ec3f765c1    C9        0
8330                N  001886bf-3b1f-49cd-a7fc-4a1ec3f765c1   GRN        0


In [7]:
# sample source data
somaMeta = pd.read_csv('../data/GNPC_Harmonized_Dataset_V1/SomalogicMetaV1_anonymized.csv')
somaMeta.head()

/tmp/ipykernel_649304/3545570142.py:2: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  somaMeta = pd.read_csv('../data/GNPC_Harmonized_Dataset_V1/SomalogicMetaV1_anonymized.csv')


,sample_id,contributor_code,visit,plate_id,units,anml_fraction_used_0_005,anml_fraction_used_0_5,anml_fraction_used_20,anml_fraction_used_20_s1,anml_fraction_used_20_s2,...,hyb_control_norm_scale,norm_scale_0_005,norm_scale_0_5,norm_scale_20,norm_scale_20_s1,norm_scale_20_s2,norm_scale_20_s3,row_check,sample_matrix,sample_type
0,8a0e5419-20f3-44d6-bb46-bab165bf8d05,R,1,147,-1,0.866,0.836,0.847,-1.0,-1.0,...,0.934112,1.435652,0.824065,0.722277,-1.0,-1.0,-1.0,PASS,EDTA Plasma,Sample
1,1f2178f2-83a6-46df-aec1-3dd741621068,R,2,147,-1,0.936,0.947,0.954,-1.0,-1.0,...,0.884058,0.894184,0.804804,0.841661,-1.0,-1.0,-1.0,PASS,EDTA Plasma,Sample
2,40c3925c-a770-4adc-9a87-5d9e2a23b855,R,1,147,-1,0.973,0.939,0.913,-1.0,-1.0,...,0.906697,1.100648,1.043171,0.901319,-1.0,-1.0,-1.0,PASS,EDTA Plasma,Sample
3,9de3c941-b32b-4ef8-8d59-6b57b905c657,R,2,147,-1,0.888,0.843,0.859,-1.0,-1.0,...,0.994137,1.165921,0.942466,0.825704,-1.0,-1.0,-1.0,PASS,EDTA Plasma,Sample
4,03830809-3976-4f58-8279-ca2ffd0398c4,R,2,147,-1,0.850,0.828,0.804,-1.0,-1.0,...,0.925489,1.061411,0.659705,0.608147,-1.0,-1.0,-1.0,PASS,EDTA Plasma,Sample


In [8]:
sample_duplicates_test(somaMeta)

In [9]:
# clinical subject data
clinicalData = pd.read_csv('../data/GNPC_Harmonized_Dataset_V1/ClinicalV1_anonymized.csv')
clinicalData.head()

,contributor_code,sample_id,sequential_visit_number,age_at_visit,computed_age_range,sex,race,years_of_education,computed_years_education_range,computed_height_range,...,anxiety,cdr,cognitive_test_date,cognitive_test_score,computed_cognitive_test_score,cognitive_test_battery,computed_clinical_diagnosis,computed_cognitive_impairment,is_neuropath,is_biomarker
0,A,2d5cc071-c66c-4a10-b55b-00d6eb03b6f3,1,69,0,1,5,14,0,0,...,0,-1.0,8/25/2021,30,1,MMSE,-1,0,0,0
1,A,26ac68a0-91e9-4d7e-8b58-00e737c3bcd7,1,76,0,1,5,4,2,0,...,0,-1.0,9/23/2022,23,1,MMSE,1,1,0,0
2,A,28f4a373-ba8e-4832-8485-0158dfd8c62b,1,65,0,2,2,14,0,0,...,1,-1.0,1/13/2022,20,1,MMSE,1,1,0,0
3,A,64b27523-4353-420e-b01a-024c470ccf90,1,65,0,2,2,14,0,0,...,0,-1.0,12/10/2021,26,1,MMSE,2,1,0,0
4,A,b68baa17-b013-412d-bf71-0254b6945e8d,1,73,0,2,2,18,0,0,...,0,-1.0,9/28/2021,27,1,MMSE,-1,0,0,0


In [10]:
sample_duplicates_test(clinicalData)

In [11]:
# clinical subject to sample mapping data
mappingData = pd.read_csv('../data/GNPC_Harmonized_Dataset_V1/PersonMappingV1_anonymized.csv')
mappingData.head()

,person_id,sample_id,is_somalogic,is_mass_spec
0,8aee12c9-962e-48f4-9d15-0008e4443c9d,8aee12c9-962e-48f4-9d15-0008e4443c9d,1,0
1,cd284063-9250-4149-9e8a-0009a615b59d,cd284063-9250-4149-9e8a-0009a615b59d,1,0
2,20fb319a-13a8-4284-83f6-000d1942beb2,20fb319a-13a8-4284-83f6-000d1942beb2,1,0
3,c0493c35-5a4f-4b34-8bdd-000e0c492ae1,c0493c35-5a4f-4b34-8bdd-000e0c492ae1,1,0
4,c0493c35-5a4f-4b34-8bdd-000e0c492ae1,7cdd93db-f393-4a53-a9cb-7cd67ef4b60b,1,0


In [12]:
sample_duplicates_test(mappingData)

In [13]:
# Separate different sample matrices into different dataframes
somaGroups = {key: subdf for key, subdf in somaMeta.groupby("sample_matrix")}
plasmaSoma = somaGroups["EDTA Plasma"]
plasmaSoma.head()

,sample_id,contributor_code,visit,plate_id,units,anml_fraction_used_0_005,anml_fraction_used_0_5,anml_fraction_used_20,anml_fraction_used_20_s1,anml_fraction_used_20_s2,...,hyb_control_norm_scale,norm_scale_0_005,norm_scale_0_5,norm_scale_20,norm_scale_20_s1,norm_scale_20_s2,norm_scale_20_s3,row_check,sample_matrix,sample_type
0,8a0e5419-20f3-44d6-bb46-bab165bf8d05,R,1,147,-1,0.866,0.836,0.847,-1.0,-1.0,...,0.934112,1.435652,0.824065,0.722277,-1.0,-1.0,-1.0,PASS,EDTA Plasma,Sample
1,1f2178f2-83a6-46df-aec1-3dd741621068,R,2,147,-1,0.936,0.947,0.954,-1.0,-1.0,...,0.884058,0.894184,0.804804,0.841661,-1.0,-1.0,-1.0,PASS,EDTA Plasma,Sample
2,40c3925c-a770-4adc-9a87-5d9e2a23b855,R,1,147,-1,0.973,0.939,0.913,-1.0,-1.0,...,0.906697,1.100648,1.043171,0.901319,-1.0,-1.0,-1.0,PASS,EDTA Plasma,Sample
3,9de3c941-b32b-4ef8-8d59-6b57b905c657,R,2,147,-1,0.888,0.843,0.859,-1.0,-1.0,...,0.994137,1.165921,0.942466,0.825704,-1.0,-1.0,-1.0,PASS,EDTA Plasma,Sample
4,03830809-3976-4f58-8279-ca2ffd0398c4,R,2,147,-1,0.850,0.828,0.804,-1.0,-1.0,...,0.925489,1.061411,0.659705,0.608147,-1.0,-1.0,-1.0,PASS,EDTA Plasma,Sample


In [14]:
def merge_left(df1, df2, id_col="sample_id"):
    merged = df1.merge(df2, on=id_col, how="left", suffixes=("", "_dup"))
    dup_cols = [c for c in merged.columns if c.endswith("_dup")]
    return merged.drop(columns=dup_cols)

def merge_meta(selected_meta, mapping, clinicalData):
    merged_with_mapping = merge_left(selected_meta, mapping, id_col="sample_id")
    merged_with_clinical = merge_left(merged_with_mapping, clinicalData, id_col="sample_id")
    
    # Check if the two columns exist
    if "visit" not in merged_with_clinical.columns or "sequential_visit_number" not in merged_with_clinical.columns:
        pass

    # Check if values are identical
    elif (merged_with_clinical["visit"] == merged_with_clinical["sequential_visit_number"]).all():
        # Drop sequential_visit_number if identical
        merged_with_clinical = merged_with_clinical.drop(columns=["sequential_visit_number"])
        print("Dropped 'sequential_visit_number' because it is identical to 'visit'.")
    else:
        # Raise error if they differ
        mismatched_rows = merged_with_clinical[merged_with_clinical["visit"] != merged_with_clinical["sequential_visit_number"]]
        raise ValueError(
            f"'visit' and 'sequential_visit_number' differ in {len(mismatched_rows)} rows."
        )
    return merged_with_clinical
plasmaSomaMerged = merge_meta(plasmaSoma, mappingData, clinicalData)
plasmaSomaMerged.head()

Dropped 'sequential_visit_number' because it is identical to 'visit'.


,sample_id,contributor_code,visit,plate_id,units,anml_fraction_used_0_005,anml_fraction_used_0_5,anml_fraction_used_20,anml_fraction_used_20_s1,anml_fraction_used_20_s2,...,anxiety,cdr,cognitive_test_date,cognitive_test_score,computed_cognitive_test_score,cognitive_test_battery,computed_clinical_diagnosis,computed_cognitive_impairment,is_neuropath,is_biomarker
0,8a0e5419-20f3-44d6-bb46-bab165bf8d05,R,1,147,-1,0.866,0.836,0.847,-1.0,-1.0,...,-1,-1.0,1/1/1900,28,1,MMSE,-1,0,1,0
1,1f2178f2-83a6-46df-aec1-3dd741621068,R,2,147,-1,0.936,0.947,0.954,-1.0,-1.0,...,-1,-1.0,1/1/1900,28,1,MMSE,-1,0,1,0
2,40c3925c-a770-4adc-9a87-5d9e2a23b855,R,1,147,-1,0.973,0.939,0.913,-1.0,-1.0,...,-1,-1.0,1/1/1900,30,1,MMSE,-1,0,1,0
3,9de3c941-b32b-4ef8-8d59-6b57b905c657,R,2,147,-1,0.888,0.843,0.859,-1.0,-1.0,...,-1,-1.0,1/1/1900,28,1,MMSE,-1,0,1,0
4,03830809-3976-4f58-8279-ca2ffd0398c4,R,2,147,-1,0.850,0.828,0.804,-1.0,-1.0,...,-1,-1.0,1/1/1900,28,1,MMSE,-1,0,1,0


In [15]:
def sample_summary(df, cols = None):
    if cols is None:
        cols = ["person_id", "sample_id", "sex", "contributor_code", "sample_type", "visit", "age_at_visit", "ad", "ftd", "pd", "als", "mci_sci", "recruited_control", "cdr", "depression", "anxiety", "cognitive_test_date", "cognitive_test_score", "computed_cognitive_test_score", "cognitive_test_battery", "computed_clinical_diagnosis", "computed_cognitive_impairment", "is_neuropath", "is_biomarker"]
    subdf = df[cols]
    return subdf

plasmaSomaSummary = sample_summary(plasmaSomaMerged)
plasmaSomaSummary.head()

,person_id,sample_id,sex,contributor_code,sample_type,visit,age_at_visit,ad,ftd,pd,...,depression,anxiety,cognitive_test_date,cognitive_test_score,computed_cognitive_test_score,cognitive_test_battery,computed_clinical_diagnosis,computed_cognitive_impairment,is_neuropath,is_biomarker
0,8a0e5419-20f3-44d6-bb46-bab165bf8d05,8a0e5419-20f3-44d6-bb46-bab165bf8d05,2,R,Sample,1,88,0,-1,0,...,-1,-1,1/1/1900,28,1,MMSE,-1,0,1,0
1,a398ecfd-c26b-4e4b-8ca0-9915be6bb958,1f2178f2-83a6-46df-aec1-3dd741621068,1,R,Sample,2,72,0,-1,0,...,-1,-1,1/1/1900,28,1,MMSE,-1,0,1,0
2,40c3925c-a770-4adc-9a87-5d9e2a23b855,40c3925c-a770-4adc-9a87-5d9e2a23b855,2,R,Sample,1,77,0,-1,0,...,-1,-1,1/1/1900,30,1,MMSE,-1,0,1,0
3,8a0e5419-20f3-44d6-bb46-bab165bf8d05,9de3c941-b32b-4ef8-8d59-6b57b905c657,2,R,Sample,2,90,0,-1,0,...,-1,-1,1/1/1900,28,1,MMSE,-1,0,1,0
4,92c1caea-95f8-4740-a038-04a85f106b62,03830809-3976-4f58-8279-ca2ffd0398c4,2,R,Sample,2,90,0,-1,0,...,-1,-1,1/1/1900,28,1,MMSE,-1,0,1,0


In [16]:
def clean_stage_noise(pf, stage_cols):
    sorted_pf = pf.sort_values(by="visit")
    for stage_col in stage_cols:
        # get rid of 0 and -1 after 1
        first_confirmed_cu = 0
        first_confirmed_ci = 0
        if 1 in sorted_pf[stage_col].values:
            first_confirmed_ci = sorted_pf.loc[sorted_pf[stage_col] == 1, "visit"].iloc[0]
            sorted_pf.loc[sorted_pf["visit"] > first_confirmed_ci, stage_col] = 1
        # get rid of -1 before 0
        if 0 in sorted_pf[stage_col].values:
            first_confirmed_cu = sorted_pf.loc[sorted_pf[stage_col] == 0, "visit"].iloc[0]
            sorted_pf.loc[sorted_pf["visit"] < first_confirmed_cu, stage_col] = 0
        if first_confirmed_cu < first_confirmed_ci:
            sorted_pf.loc[(sorted_pf["visit"] > first_confirmed_cu) & (sorted_pf["visit"] < first_confirmed_ci), stage_col] = 0

    return sorted_pf



In [17]:
def converter_check(pdf):
    converter_status = None
    person_df = clean_stage_noise(pdf, stage_cols=['mci_sci', 'ad'])

    mci_vals = person_df['mci_sci'].unique()
    ad_vals = person_df['ad'].unique()

    if 0 in mci_vals and 1 in mci_vals and 0 in ad_vals and 1 in ad_vals:
        converter_status = "cu2mci2ad"
    elif 0 in mci_vals and 1 in mci_vals and 1 not in ad_vals:
        converter_status = "cu2mci"
    elif 0 in ad_vals and 1 in ad_vals:
        converter_status = "mci2ad"
    elif (1 in mci_vals and 0 not in mci_vals and 1 not in ad_vals):
        converter_status = "mci"
    elif (1 in ad_vals and 0 not in ad_vals):
        converter_status = "ad"
    else:
        converter_status = "cu"
    return converter_status

In [18]:
def other_nd_check(person_df, nd_type):
    nd_status = None
    nd_vals = person_df[nd_type].unique()

    if 0 in nd_vals and 1 in nd_vals:
        nd_status = f"cu2{nd_type}"
    elif (1 in nd_vals and 0 not in nd_vals):
        nd_status = nd_type
    else:
        nd_status = "cu"
    return nd_status

In [19]:
def continuous_summary(patients, disease_stage, person_dfs, metric):
    dfs = [person_dfs.get_group(pid) for pid in patients[disease_stage][-1] + patients[disease_stage][1] + patients[disease_stage][2]]
    combined_df = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
    if combined_df.empty:
        return {stage: {} for stage in disease_stage.split("2")}
    stages = disease_stage.split("2")
    summary = {}
    for stage in stages:
        if stage == "cu":
            stage_df = combined_df[(combined_df['ad'] == 0) & (combined_df['mci_sci'] == 0)]
        elif stage == "mci":
            stage_df = combined_df[(combined_df['mci_sci'] == 1) & (combined_df['ad'] == 0)]
        elif stage == "ad":
            stage_df = combined_df[combined_df['ad'] == 1]
        
        if metric == "cognitive_test_score":
            # metric_counts = stage_df.loc[stage_df[metric].between(0, 30), metric].describe().to_dict()
            df = stage_df.loc[stage_df[metric].between(0, 30)].copy()
            if df.empty:
                weighted_mean = np.nan
                weighted_std = np.nan
            else:
                df['weight'] = (
                    1 / df.groupby('person_id')[metric].transform('count')
                )

                weighted_mean = np.average(df[metric], weights=df['weight'])
                weighted_var = np.average(
                    (df[metric] - weighted_mean)**2,
                    weights=df['weight']
                )
                weighted_std = np.sqrt(weighted_var)
        else:
            raise ValueError(f"Unsupported continuous metric: {metric}")
        # summary[stage] = metric_counts
        summary[stage] = {
            'weighted_mean': weighted_mean,
            'weighted_std': weighted_std
        }
    return summary


In [20]:
def discrete_status_approximate(pf, status_cols):
    for stage in ["cu", "mci", "ad"]:
        for status_col in status_cols:
            if stage == "cu":
                stage_df = pf[(pf['ad'] == 0) & (pf['mci_sci'] == 0)]
            elif stage == "mci":
                stage_df = pf[(pf['mci_sci'] == 1) & (pf['ad'] == 0)]
            elif stage == "ad":
                stage_df = pf[pf['ad'] == 1]
        first_confirmed_cu = 0
        first_confirmed_ci = 0
        if 1 in pf[status_col].values:
            first_confirmed_ci = pf.loc[pf[status_col] == 1, "visit"].iloc[0]
            pf.loc[pf["visit"] > first_confirmed_ci, status_col] = 1
        # get rid of -1 before 0
        if 0 in pf[status_col].values:
            first_confirmed_cu = pf.loc[pf[status_col] == 0, "visit"].iloc[0]
            pf.loc[pf["visit"] < first_confirmed_cu, status_col] = 0
        if first_confirmed_cu < first_confirmed_ci:
            pf.loc[(pf["visit"] > first_confirmed_cu) & (pf["visit"] < first_confirmed_ci), status_col] = 0

    return pf

In [21]:
def discrete_summary(patients, disease_stage, person_dfs, metric):
    dfs = [person_dfs.get_group(pid) for pid in patients[disease_stage][-1] + patients[disease_stage][1] + patients[disease_stage][2]]
    combined_df = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
    if combined_df.empty:
        return {stage: {} for stage in disease_stage.split("2")}
    stages = disease_stage.split("2")
    summary = {}
    for stage in stages:
        if stage == "cu":
            stage_df = combined_df[(combined_df['ad'] == 0) & (combined_df['mci_sci'] == 0)]
        elif stage == "mci":
            stage_df = combined_df[(combined_df['mci_sci'] == 1) & (combined_df['ad'] == 0)]
        elif stage == "ad":
            stage_df = combined_df[combined_df['ad'] == 1]
        
        metric_counts = dict.fromkeys(stage_df[metric].unique(), 0)
        for pid in stage_df['person_id'].unique():
            person_df = stage_df[stage_df['person_id'] == pid]
            vals = person_df[metric].unique()
            for val in vals:
                metric_counts[val] += 1
        summary[stage] = metric_counts
    return summary

In [22]:
def genetic_summary(patients, disease_stage, person_dfs, geneticsData):
    dfs = [person_dfs.get_group(pid) for pid in patients[disease_stage][-1] + patients[disease_stage][1] + patients[disease_stage][2]]
    combined_df = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
    if combined_df.empty:
        return {stage: {} for stage in disease_stage.split("2")}
    stages = disease_stage.split("2")
    summary = {}
    for stage in stages:
        if stage == "cu":
            stage_df = combined_df[(combined_df['ad'] == 0) & (combined_df['mci_sci'] == 0)]
        elif stage == "mci":
            stage_df = combined_df[(combined_df['mci_sci'] == 1) & (combined_df['ad'] == 0)]
        elif stage == "ad":
            stage_df = combined_df[combined_df['ad'] == 1]
        summary[stage] = {}
        for _, row in stage_df.iterrows():
            person_id = row['person_id']
            sample_id = row['sample_id']
            sample_genetics = geneticsData[geneticsData['sample_id'] == sample_id]
            for _, gene_row in sample_genetics.iterrows():
                gene = gene_row['gene']
                variant = gene_row['variant']
                if gene in summary[stage]:
                    if variant in summary[stage][gene]:
                        summary[stage][gene][variant].add(person_id)
                    else:
                        summary[stage][gene][variant] = set([person_id])
                else:
                    summary[stage][gene] = {variant: set([person_id])}
        
    return summary

In [23]:
def cohort_summary(df, cohort, geneticsData):
    grouped = df.groupby("contributor_code")
    cohort_df = grouped.get_group(cohort)
    person_dfs = cohort_df.groupby("person_id")
    nd_types = ["pd", "ftd", "als"]
    discrete_metrics = ["depression", "anxiety", "cdr", "computed_cognitive_test_score"]
    continuous_metrics = ["cognitive_test_score"]


    single_visit_converter_patients = {"cu": {-1: [], 1: [], 2: []}, "mci": {-1: [], 1: [], 2: []}, "ad": {-1: [], 1: [], 2: []}}
    single_visit_other_nd_patients = {"pd": {-1: [], 1: [], 2: []}, "ftd": {-1: [], 1: [], 2: []}, "als": {-1: [], 1: [], 2: []}}
    multi_visit_converter_patients = {"cu": {-1: [], 1: [], 2: []}, "mci": {-1: [], 1: [], 2: []}, "ad": {-1: [], 1: [], 2: []}, "cu2mci": {-1: [], 1: [], 2: []}, "mci2ad": {-1: [], 1: [], 2: []}, "cu2ad": {-1: [], 1: [], 2: []}, "cu2mci2ad": {-1: [], 1: [], 2: []}}
    multi_visit_other_nd_patients = {"pd": {-1: [], 1: [], 2: []}, "ftd": {-1: [], 1: [], 2: []}, "als": {-1: [], 1: [], 2: []}, "cu2pd": {-1: [], 1: [], 2: []}, "cu2ftd": {-1: [], 1: [], 2: []}, "cu2als": {-1: [], 1: [], 2: []}}

    for person_id, person_df in person_dfs:
        visits = person_df['visit'].nunique()
        record_count = len(person_df)
        conv_state = converter_check(person_df)
        if (record_count != visits):
            print(f"Data inconsistency for person_id {person_id}: {record_count} records but {visits} unique visits.")
        if record_count == 1:
            single_visit_converter_patients[conv_state][person_df['sex'].iloc[0]].append(person_id)
            for nd in nd_types:
                nd_state = other_nd_check(person_df, nd)
                if nd_state != "cu":
                    single_visit_other_nd_patients[nd_state][person_df['sex'].iloc[0]].append(person_id)
        else:
            multi_visit_converter_patients[conv_state][person_df['sex'].iloc[0]].append(person_id)
            for nd in nd_types:
                nd_state = other_nd_check(person_df, nd)
                if nd_state != "cu":
                    multi_visit_other_nd_patients[nd_state][person_df['sex'].iloc[0]].append(person_id)
    
    print(f"Cohort: {cohort}")
    for state, stats in single_visit_converter_patients.items():
        print(f"Single-visit converter patients:\n {state}:\n male : {len(stats[1])}, female: {len(stats[2])}, unknown: {len(stats[-1])}, total: {len(stats[1]) + len(stats[2]) + len(stats[-1])}")

    for state, stats in multi_visit_converter_patients.items():
        print(f"Multiple-visit converter patients:\n {state}:\n male : {len(stats[1])}, female: {len(stats[2])}, unknown: {len(stats[-1])}, total: {len(stats[1]) + len(stats[2]) + len(stats[-1])}")

    for ad_state, ad_stats in single_visit_converter_patients.items():
        for nd_state, nd_stats in single_visit_other_nd_patients.items():
            print(f"Single-visit patients:\n Converter state: {ad_state}, Other ND state: {nd_state}:\n male : {len(set(ad_stats[1]) & set(nd_stats[1]))}, female: {len(set(ad_stats[2]) & set(nd_stats[2]))}, unknown: {len(set(ad_stats[-1]) & set(nd_stats[-1]))}, total: {len(set(ad_stats[1]) & set(nd_stats[1])) + len(set(ad_stats[2]) & set(nd_stats[2])) + len(set(ad_stats[-1]) & set(nd_stats[-1]))}")
    
    for ad_state, ad_stats in multi_visit_converter_patients.items():
        for nd_state, nd_stats in multi_visit_other_nd_patients.items():
            print(f"Multiple-visit patients:\n Converter state: {ad_state}, Other ND state: {nd_state}:\n male : {len(set(ad_stats[1]) & set(nd_stats[1]))}, female: {len(set(ad_stats[2]) & set(nd_stats[2]))}, unknown: {len(set(ad_stats[-1]) & set(nd_stats[-1]))}, total: {len(set(ad_stats[1]) & set(nd_stats[1])) + len(set(ad_stats[2]) & set(nd_stats[2])) + len(set(ad_stats[-1]) & set(nd_stats[-1]))}")
    
    for metric in discrete_metrics:
        for stage in single_visit_converter_patients.keys():
            summary = discrete_summary(single_visit_converter_patients, stage, person_dfs, metric)
            print(f"Single-visit patients - Discrete metric '{metric}' summary for stage '{stage}': {summary}")
        for stage in multi_visit_converter_patients.keys():
            summary = discrete_summary(multi_visit_converter_patients, stage, person_dfs, metric)
            print(f"Multiple-visit patients - Discrete metric '{metric}' summary for stage '{stage}': {summary}")
    
    for metric in continuous_metrics:
        for stage in single_visit_converter_patients.keys():
            summary = continuous_summary(single_visit_converter_patients, stage, person_dfs, metric)
            print(f"Single-visit patients - Continuous metric '{metric}' summary for stage '{stage}': {summary}")
        for stage in multi_visit_converter_patients.keys():
            summary = continuous_summary(multi_visit_converter_patients, stage, person_dfs, metric)
            print(f"Multiple-visit patients - Continuous metric '{metric}' summary for stage '{stage}': {summary}")
    
    for stage in single_visit_converter_patients.keys():
        summary = genetic_summary(single_visit_converter_patients, stage, person_dfs, geneticsData)
        print(f"Single-visit patients - Genetic summary for stage '{stage}':")
        for substage, gene_data in summary.items():
            print(f"  Stage: {substage}")
            for gene, variants in gene_data.items():
                for variant, persons in variants.items():
                    print(f"    Gene: {gene}, Variant: {variant}, Count: {len(persons)}")

    for stage in multi_visit_converter_patients.keys():
        summary = genetic_summary(multi_visit_converter_patients, stage, person_dfs, geneticsData)
        print(f"Multiple-visit patients - Genetic summary for stage '{stage}':")
        for substage, gene_data in summary.items():
            print(f"  Stage: {substage}")
            for gene, variants in gene_data.items():
                for variant, persons in variants.items():
                    print(f"    Gene: {gene}, Variant: {variant}, Count: {len(persons)}")
    return single_visit_converter_patients, single_visit_other_nd_patients, multi_visit_converter_patients, multi_visit_other_nd_patients

In [24]:
for cohort in ["B", "D", "F", "I", "J", "P", "R"]:
    cohort_summary(plasmaSomaSummary, cohort, geneticsData)

Cohort: B
Single-visit converter patients:
 cu:
 male : 0, female: 0, unknown: 0, total: 0
Single-visit converter patients:
 mci:
 male : 0, female: 0, unknown: 0, total: 0
Single-visit converter patients:
 ad:
 male : 0, female: 0, unknown: 0, total: 0
Multiple-visit converter patients:
 cu:
 male : 101, female: 143, unknown: 0, total: 244
Multiple-visit converter patients:
 mci:
 male : 37, female: 48, unknown: 0, total: 85
Multiple-visit converter patients:
 ad:
 male : 0, female: 0, unknown: 0, total: 0
Multiple-visit converter patients:
 cu2mci:
 male : 38, female: 76, unknown: 0, total: 114
Multiple-visit converter patients:
 mci2ad:
 male : 0, female: 0, unknown: 0, total: 0
Multiple-visit converter patients:
 cu2ad:
 male : 0, female: 0, unknown: 0, total: 0
Multiple-visit converter patients:
 cu2mci2ad:
 male : 0, female: 0, unknown: 0, total: 0
Single-visit patients:
 Converter state: cu, Other ND state: pd:
 male : 0, female: 0, unknown: 0, total: 0
Single-visit patients:
 C

In [25]:
rlt = cohort_summary(plasmaSomaSummary, "P", geneticsData)

Cohort: P
Single-visit converter patients:
 cu:
 male : 12, female: 3, unknown: 0, total: 15
Single-visit converter patients:
 mci:
 male : 2, female: 1, unknown: 0, total: 3
Single-visit converter patients:
 ad:
 male : 0, female: 0, unknown: 0, total: 0
Multiple-visit converter patients:
 cu:
 male : 203, female: 185, unknown: 0, total: 388
Multiple-visit converter patients:
 mci:
 male : 16, female: 9, unknown: 0, total: 25
Multiple-visit converter patients:
 ad:
 male : 0, female: 0, unknown: 0, total: 0
Multiple-visit converter patients:
 cu2mci:
 male : 55, female: 33, unknown: 0, total: 88
Multiple-visit converter patients:
 mci2ad:
 male : 0, female: 0, unknown: 0, total: 0
Multiple-visit converter patients:
 cu2ad:
 male : 0, female: 0, unknown: 0, total: 0
Multiple-visit converter patients:
 cu2mci2ad:
 male : 0, female: 0, unknown: 0, total: 0
Single-visit patients:
 Converter state: cu, Other ND state: pd:
 male : 0, female: 0, unknown: 0, total: 0
Single-visit patients:
 C

In [26]:
stage_patients = []
for gender, pid in rlt[2]["cu2mci"].items():
    stage_patients.extend(pid)

In [27]:
stage_data = plasmaSomaSummary[plasmaSomaSummary["person_id"].isin(stage_patients)]
patient_level = stage_data.groupby("person_id")
count = 0
for pid, pdf in patient_level:
    count += 1
    print(f"Person ID: {pid}, index: {count}")
    print(pdf[["visit", "mci_sci", "ad", "ftd", "als", "pd", "recruited_control"]].sort_values(by="visit"))

Person ID: 05efe196-294b-44b3-bb34-77aeabc4297c, index: 1
       visit  mci_sci  ad  ftd  als  pd  recruited_control
15996      1        0  -1   -1   -1  -1                  2
15070      2        0  -1   -1   -1  -1                  2
15337      3        0  -1   -1   -1  -1                  2
13865      4        1  -1   -1   -1  -1                  2
Person ID: 06204f24-20be-4939-a8e1-2c9a6d6b4239, index: 2
       visit  mci_sci  ad  ftd  als  pd  recruited_control
17346      1        0  -1   -1   -1  -1                  2
16282      2        0  -1   -1   -1  -1                  2
17347      3        1  -1   -1   -1  -1                  2
Person ID: 0f55b2f8-a793-4dfc-9b7d-da1a71731215, index: 3
       visit  mci_sci  ad  ftd  als  pd  recruited_control
14819      1        0  -1   -1   -1  -1                  2
12194      2        0  -1   -1   -1  -1                  2
15611      3        0  -1   -1   -1  -1                  2
12153      4        1  -1   -1   -1  -1                  2


In [28]:
cohort_p = pd.read_csv('../data/GNPC_Harmonized_Dataset_V1/contributor_p.csv')
cohort_p.head()

,sample_id,person_id,contributor_code,visit,sample_type,row_check,sample_matrix,ad,ftd,mci_sci,...,EMILIN3_1,ZNF264,ATP4B,DUT,UBXN4_1,IRF6,Status,Diagnosis,Converters,Progression_Simple
0,00683dc0-4d6a-4933-a2c4-0bab1c5be01e,00683dc0-4d6a-4933-a2c4-0bab1c5be01e,P,1,Sample,PASS,EDTA Plasma,-1,-1,0,...,12.338820,10.019591,10.370033,11.907003,14.432633,10.963691,CU,CU,Non-Converter,CU
1,c50241d8-8416-40af-a362-266a4f3c7145,00683dc0-4d6a-4933-a2c4-0bab1c5be01e,P,2,Sample,PASS,EDTA Plasma,-1,-1,0,...,11.697619,10.014439,10.520815,10.080151,12.000247,10.069315,CU,CU,Non-Converter,CU
2,3251f185-b759-4d58-a12b-35158f081011,00683dc0-4d6a-4933-a2c4-0bab1c5be01e,P,3,Sample,PASS,EDTA Plasma,-1,-1,0,...,10.096583,10.295884,10.477455,13.033062,15.248561,11.991593,CU,CU,Non-Converter,CU
3,016e274b-6fe0-436f-a7d0-64a372618828,016e274b-6fe0-436f-a7d0-64a372618828,P,1,Sample,PASS,EDTA Plasma,-1,-1,0,...,9.068510,10.342519,10.529040,13.074292,15.543687,11.966758,CU,CU,Non-Converter,CU
4,3adf0c5c-ec0a-42bb-9304-e53d6782693e,016e274b-6fe0-436f-a7d0-64a372618828,P,2,Sample,FLAG,EDTA Plasma,-1,-1,0,...,9.278682,10.564435,10.558803,13.627944,15.639835,12.571516,CU,CU,Non-Converter,CU


In [29]:
cohort_p["Converters"].unique()

array(['Non-Converter', 'Converter'], dtype=object)

In [56]:
cohort_p["Progression_Simple"].unique()

array(['CU', 'MCI/SCI → Other', 'CU → MCI/SCI', 'Other', 'CU → Other',
       'MCI/SCI', 'CU → MCI/SCI → Other', 'Other → MCI/SCI → Other',
       'MCI/SCI → Other → MCI/SCI', 'Other → MCI/SCI',
       'Other → MCI/SCI → Other → MCI/SCI'], dtype=object)

In [43]:
p1 = list(cohort_p[(cohort_p["Converters"] == "Converter") & (cohort_p["Progression_Simple"] == "CU → MCI/SCI") ]["person_id"].unique())

In [44]:
p2 = list(cohort_p[(cohort_p["Converters"] == "Converter") & (cohort_p["Progression_Simple"] == "CU → MCI/SCI → Other")]["person_id"].unique())

In [46]:
aatman_patients = p1 + p2
aatman_patients = set(aatman_patients)
len(aatman_patients)

81

In [48]:
my_patients = set(stage_patients)
len(my_patients)

88

In [49]:
inter  = aatman_patients & my_patients
len(inter)

81

In [ ]:
addition = my_patients - inter
count = 0
for pid, pdf in patient_level:
    if pid in addition:
        count += 1
        print(f"Person ID: {pid}, index: {count}")
        print(pdf[["visit", "mci_sci", "ad", "ftd", "als", "pd", "recruited_control"]].sort_values(by="visit"))

Person ID: 4a13057c-cec6-47af-b923-a5568e936121, index: 1
       visit  mci_sci  ad  ftd  als  pd  recruited_control
14518      1        0  -1   -1   -1  -1                  2
14521      2        1  -1   -1   -1  -1                  2
14817      3        0  -1   -1   -1  -1                  2
Person ID: ab3eb53e-7a89-4a49-ac74-f1ed149e2fbf, index: 2
       visit  mci_sci  ad  ftd  als  pd  recruited_control
17299      1        0  -1   -1   -1  -1                  2
17300      2        1  -1   -1   -1  -1                  2
16080      3        1  -1   -1   -1  -1                  2
Person ID: cd7f2292-a2bf-4087-a378-1bda40488315, index: 3
       visit  mci_sci  ad  ftd  als  pd  recruited_control
16017      1        0  -1   -1   -1  -1                  2
15442      2        1  -1   -1   -1  -1                  2
Person ID: ed38723a-d3f9-4447-bdd8-b320237f6644, index: 4
       visit  mci_sci  ad  ftd  als  pd  recruited_control
15338      1        0  -1   -1   -1  -1                  2
1

In [58]:
p3 = list(cohort_p[(cohort_p["Converters"] == "Non-Converter") & (cohort_p["Status"] == "CU") ]["person_id"].unique())
len(p3)

360

In [ ]:
stage_patients3 = []
for gender, pid in rlt[2]["cu"].items():
    stage_patients3.extend(pid)
len(stage_patients3)

388

In [61]:
inter3 = set(p3) & set(stage_patients3)
len(inter3)

360

In [68]:
addition3 = set(stage_patients3) - inter3
stage_data3 = plasmaSomaSummary[plasmaSomaSummary["person_id"].isin(stage_patients3)]
patient_level3 = stage_data3.groupby("person_id")
count = 0
for pid, pdf in patient_level3:
    if pid in addition3:
        count += 1
        print(f"Person ID: {pid}, index: {count}")
        print(pdf[["visit", "mci_sci", "ad", "ftd", "als", "pd", "recruited_control"]].sort_values(by="visit"))

Person ID: 078768db-e696-477a-964c-8e80291d3a17, index: 1
       visit  mci_sci  ad  ftd  als  pd  recruited_control
16951      1        0  -1   -1   -1  -1                  2
15858      2        0  -1   -1   -1  -1                  2
Person ID: 07c7a253-021b-4dfe-aa27-400fe3a1b59e, index: 2
       visit  mci_sci  ad  ftd  als  pd  recruited_control
16083      1        0  -1   -1   -1  -1                  2
16655      2        0  -1   -1   -1  -1                  2
16656      3        0  -1   -1   -1  -1                  2
16084      4        0  -1   -1   -1  -1                  2
Person ID: 36c995ac-16cf-4108-b94d-b66c83742432, index: 3
       visit  mci_sci  ad  ftd  als  pd  recruited_control
16915      1        0  -1   -1   -1  -1                  2
15989      2        0  -1   -1   -1  -1                  2
Person ID: 39f71c3f-4da8-46b5-b438-ed276fdf9663, index: 4
       visit  mci_sci  ad  ftd  als  pd  recruited_control
15463      1        0  -1   -1   -1  -1                  2
1

In [57]:
p4 = list(cohort_p[(cohort_p["Converters"] == "Non-Converter") & (cohort_p["Status"] == "CI") ]["person_id"].unique())
len(p4)

40

In [60]:
stage_patients4 = []
for gender, pid in rlt[2]["mci"].items():
    stage_patients4.extend(pid)
len(stage_patients4)

25

In [69]:
inter4 = set(p4) & set(stage_patients4)
len(inter4)

25

In [75]:
addition4 = set(p4) - inter4
stage_data4 = plasmaSomaSummary
patient_level4 = stage_data4.groupby("person_id")
count = 0
for pid, pdf in patient_level4:
    if pid in addition4:
        count += 1
        print(f"Person ID: {pid}, index: {count}")
        print(converter_check(pdf))
        print(pdf[["visit", "mci_sci", "ad", "ftd", "als", "pd", "recruited_control"]].sort_values(by="visit"))

Person ID: 078768db-e696-477a-964c-8e80291d3a17, index: 1
cu
       visit  mci_sci  ad  ftd  als  pd  recruited_control
16951      1        0  -1   -1   -1  -1                  2
15858      2        0  -1   -1   -1  -1                  2
Person ID: 36c995ac-16cf-4108-b94d-b66c83742432, index: 2
cu
       visit  mci_sci  ad  ftd  als  pd  recruited_control
16915      1        0  -1   -1   -1  -1                  2
15989      2        0  -1   -1   -1  -1                  2
Person ID: 4a13057c-cec6-47af-b923-a5568e936121, index: 3
cu2mci
       visit  mci_sci  ad  ftd  als  pd  recruited_control
14518      1        0  -1   -1   -1  -1                  2
14521      2        1  -1   -1   -1  -1                  2
14817      3        0  -1   -1   -1  -1                  2
Person ID: 5b226e3f-bffd-4c12-ad52-0ca14b7723bb, index: 4
cu
       visit  mci_sci  ad  ftd  als  pd  recruited_control
17473      1        0  -1   -1   -1  -1                  2
16782      2        0  -1   -1   -1  -1     

In [31]:
cohort_p["Status"].unique()

array(['CU', 'CI'], dtype=object)

In [32]:
cohort_p["Diagnosis"].unique()

array(['CU', 'MCI/SCI', 'Other'], dtype=object)

In [33]:
cohort_p["Progression_Simple"].unique()

array(['CU', 'MCI/SCI → Other', 'CU → MCI/SCI', 'Other', 'CU → Other',
       'MCI/SCI', 'CU → MCI/SCI → Other', 'Other → MCI/SCI → Other',
       'MCI/SCI → Other → MCI/SCI', 'Other → MCI/SCI',
       'Other → MCI/SCI → Other → MCI/SCI'], dtype=object)